In [0]:
%sql
USE CATALOG `retail-sales-data-warehouse`;

CREATE SCHEMA IF NOT EXISTS validation;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS validation.validation_report (

    validation_id BIGINT GENERATED ALWAYS AS IDENTITY,

    layer_name STRING,
    table_name STRING,

    validation_type STRING,
    validation_status STRING,

    failed_records BIGINT,

    remarks STRING,

    validation_time TIMESTAMP

)

USING DELTA;

In [0]:
%sql
INSERT INTO validation.validation_report (

    layer_name,
    table_name,

    validation_type,
    validation_status,

    failed_records,
    remarks,

    validation_time

)

SELECT

    'SILVER',
    'customers_silver',

    'NULL_CHECK',

    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,

    COUNT(*),

    'Checking NULL customer IDs',

    CURRENT_TIMESTAMP()

FROM silver.customers_silver

WHERE customer_id IS NULL;

In [0]:
%sql
INSERT INTO validation.validation_report (

    layer_name,
    table_name,

    validation_type,
    validation_status,

    failed_records,
    remarks,

    validation_time

)

SELECT

    'SILVER',
    'customers_silver',

    'DUPLICATE_CHECK',

    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,

    COUNT(*),

    'Checking duplicate customer IDs',

    CURRENT_TIMESTAMP()

FROM (

    SELECT
        customer_id,
        COUNT(*) cnt

    FROM silver.customers_silver

    GROUP BY customer_id

    HAVING COUNT(*) > 1

);

In [0]:
%sql
INSERT INTO validation.validation_report (

    layer_name,
    table_name,

    validation_type,
    validation_status,

    failed_records,
    remarks,

    validation_time

)

SELECT

    'SILVER',
    'sales_silver',

    'NEGATIVE_QUANTITY_CHECK',

    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,

    COUNT(*),

    'Checking invalid sales quantity',

    CURRENT_TIMESTAMP()

FROM silver.sales_silver

WHERE quantity <= 0;

In [0]:
%sql
INSERT INTO validation.validation_report (

    layer_name,
    table_name,

    validation_type,
    validation_status,

    failed_records,
    remarks,

    validation_time

)

SELECT

    'GOLD',
    'dim_customer',

    'ROW_COUNT_VALIDATION',

    CASE
        WHEN gold_count >= silver_count THEN 'PASS'
        ELSE 'FAIL'
    END,

    ABS(gold_count - silver_count),

    'Comparing Silver vs Gold customer counts',

    CURRENT_TIMESTAMP()

FROM (

    SELECT
        (SELECT COUNT(*) FROM gold.dim_customer WHERE is_active = 1) AS gold_count,
        (SELECT COUNT(*) FROM silver.customers_silver) AS silver_count

);

In [0]:
%sql
SELECT *
FROM validation.validation_report
ORDER BY validation_time DESC;